# BraTS Inference + Evaluation Pipeline
Runs inference, generates reference reports from BTReport metadata, and evaluates on a T4 GPU.

### Before running:
1. Upload your `Brats` folder to Google Drive (including `brats_data/`, `frames/`, and all `.py` files)
2. Set `Runtime > Change runtime type > T4 GPU`
3. Run all cells in order — no API keys required

## 1. Mount Google Drive

In [1]:
!pip install "tokenizers>=0.22.0" -q

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2. Set paths
Update `DRIVE_BRATS_DIR` to match where your `Brats` folder lives in Drive.

In [4]:
import os

# --- EDIT THIS if your Brats folder is in a subfolder of Drive ---
DRIVE_BRATS_DIR = "/content/drive/MyDrive/Brats"

# Working directory in Colab local storage (faster I/O than Drive)
WORK_DIR = "/content/Brats"

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Drive path : {DRIVE_BRATS_DIR}")
print(f"Work dir   : {WORK_DIR}")

Drive path : /content/drive/MyDrive/Brats
Work dir   : /content/Brats


## 3. Copy data from Drive to Colab local storage
Local storage has much faster I/O than Drive — this avoids bottlenecks during inference.

In [5]:
import shutil
from pathlib import Path

def sync_from_drive(src_name):
    src = Path(DRIVE_BRATS_DIR) / src_name
    dst = Path(WORK_DIR) / src_name
    if not src.exists():
        print(f"  SKIP {src_name} — not found in Drive")
        return
    if dst.exists():
        print(f"  SKIP {src_name} — already in local storage")
        return
    print(f"  Copying {src_name} from Drive …")
    if src.is_dir():
        shutil.copytree(str(src), str(dst))
    else:
        shutil.copy2(str(src), str(dst))
    print(f"  Done: {dst}")

for item in ["brats_data", "frames", "reports",
             "run_inference.py", "match_reports.py", "evaluate.py", "requirements.txt"]:
    sync_from_drive(item)

# Ensure output dirs exist
for d in ["reports", "frames", "reference_reports", "evaluation"]:
    Path(WORK_DIR, d).mkdir(exist_ok=True)

print("\nLocal Brats directory:")
!ls /content/Brats/

  Copying brats_data from Drive …
  Done: /content/Brats/brats_data
  SKIP frames — not found in Drive
  SKIP reports — not found in Drive
  Copying run_inference.py from Drive …
  Done: /content/Brats/run_inference.py
  Copying match_reports.py from Drive …
  Done: /content/Brats/match_reports.py
  Copying evaluate.py from Drive …
  Done: /content/Brats/evaluate.py
  Copying requirements.txt from Drive …
  Done: /content/Brats/requirements.txt

Local Brats directory:
brats_data   evaluation  match_reports.py   reports	      run_inference.py
evaluate.py  frames	 reference_reports  requirements.txt


## 4. Install dependencies

In [6]:
!pip install -q \
    transformers>=4.47.0 accelerate qwen-vl-utils \
    nibabel Pillow numpy torch torchvision \
    requests \
    rouge-score nltk bert-score pandas
print("Dependencies installed.")

Dependencies installed.


## 5. Check GPU

In [7]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


## 6. Run inference
Skips patients that already have a report in `reports/`. If you already ran this on Colab and synced reports back to Drive, this cell will skip all 5 patients.

In [8]:
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")
%run run_inference.py

Working directory: /content/Brats
Found 5 patient(s) under brats_data/
Using CUDA (Tesla T4) with bfloat16.
Loading Qwen/Qwen2.5-VL-3B-Instruct …  (first run downloads ~1.6 GB from HuggingFace)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Model loaded on cuda:0.

  Patient: BraTS-GLI-00000-000
  Packing MRI volumes into video frames …
    [t1c]  BraTS-GLI-00000-000-t1c.nii


/content/Brats/run_inference.py:200: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(data[i], mode="L").convert("RGB")


           → 48 slices sampled
    [t1n]  BraTS-GLI-00000-000-t1n.nii
           → 48 slices sampled
    [t2f]  BraTS-GLI-00000-000-t2f.nii
           → 48 slices sampled
    [t2w]  BraTS-GLI-00000-000-t2w.nii
           → 48 slices sampled
  Total frames: 192  (≤48 per modality × 4 modalities)
  Running VLM inference …


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


    Generating report for BraTS-GLI-00000-000 …
  Report saved → reports/BraTS-GLI-00000-000.txt

  --- Preview (first 400 chars) ---
FINDINGS:
- **T1c (Frames ~1–48):**
  - A well-defined, heterogeneous mass is present in the left frontal lobe.
  - The mass shows intense ring enhancement around its periphery, indicating possible malignancy.
  - No significant central necrosis or cystic components are observed within the mass.
  - The surrounding white matter shows mild hyperintensity on T2/FLAIR (T2-FLAIR) sequences, suggesting
  ---

  Patient: BraTS-GLI-00002-000
  Packing MRI volumes into video frames …
    [t1c]  BraTS-GLI-00002-000-t1c.nii
           → 48 slices sampled
    [t1n]  BraTS-GLI-00002-000-t1n.nii
           → 48 slices sampled
    [t2f]  BraTS-GLI-00002-000-t2f.nii
           → 48 slices sampled
    [t2w]  BraTS-GLI-00002-000-t2w.nii
           → 48 slices sampled
  Total frames: 192  (≤48 per modality × 4 modalities)
  Running VLM inference …
    Generating report fo

## 7. Generate reference reports (`match_reports.py`)
Fetches `brats23_metadata-report-deepseek-r1:1.5b.json` from the BTReport GitHub repo.  
This file contains reports for all 5 of your BraTS patients generated by DeepSeek R1 1.5B  
from real quantitative imaging features (VASARI, tumor volumes, midline shift).  
No API key required.

In [9]:
import requests
from pathlib import Path

METADATA_REPORTS_URL = (
    "https://raw.githubusercontent.com/KurtLabUW/BTReport/main/"
    "btreport/llm_report_generation/example_reports/"
    "brats23_metadata-report-deepseek-r1%3A1.5b.json"
)

os.chdir(WORK_DIR)
ref_dir = Path(WORK_DIR) / "reference_reports"
ref_dir.mkdir(exist_ok=True)

print("Fetching BTReport metadata reports …")
data = requests.get(METADATA_REPORTS_URL, timeout=30).json()
print(f"Fetched {len(data)} entries.")

patient_ids = sorted(
    d.name for d in Path(WORK_DIR, "brats_data").iterdir()
    if d.is_dir() and d.name.startswith("BraTS-")
)

for pid in patient_ids:
    ref_path = ref_dir / f"{pid}.txt"
    if ref_path.exists():
        print(f"  {pid} — already exists, skipping.")
        continue
    if pid not in data:
        print(f"  {pid} — NOT FOUND in JSON")
        continue
    findings = data[pid].get("findings", "").strip()  # lowercase key
    if not findings:
        print(f"  {pid} — findings field empty")
        continue
    header = f"# Source: BTReport / DeepSeek R1 1.5B (metadata-based)\n{'#'*72}\n\n"
    ref_path.write_text(header + findings, encoding="utf-8")
    print(f"  {pid} — saved")

print(f"\nFiles saved: {[f.name for f in sorted(ref_dir.glob('*.txt'))]}")


Fetching BTReport metadata reports …
Fetched 6 entries.
  BraTS-GLI-00000-000 — saved
  BraTS-GLI-00002-000 — saved
  BraTS-GLI-00003-000 — saved
  BraTS-GLI-00005-000 — saved
  BraTS-GLI-00006-000 — saved

Files saved: ['BraTS-GLI-00000-000.txt', 'BraTS-GLI-00002-000.txt', 'BraTS-GLI-00003-000.txt', 'BraTS-GLI-00005-000.txt', 'BraTS-GLI-00006-000.txt']


## 8. Evaluate (`evaluate.py`)
Computes 7 metrics per patient (ROUGE-L, METEOR, BLEU-4, BERTScore-F1, RadGraph-F1, RaTEScore, GREEN)  
comparing `reports/` (your model's output) against `reference_reports/` (BTReport DeepSeek metadata reports).  
Results saved to `evaluation/results.csv`.

In [10]:
os.chdir(WORK_DIR)
%run evaluate.py


Evaluating 5 patient(s) …



/content/Brats/evaluate.py:190: UserWarning: Could not load YtongXie/RaTEScore (YtongXie/RaTEScore is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`).
  APPROXIMATION: RaTEScore will be computed as BERTScore-F1 with microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.
  warnings.warn(
/content/Brats/evaluate.py:251: UserWarning: Could not load StanfordMIMI/GREEN (StanfordMIMI/GREEN is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`).
  APPROXIMATION: GREEN will be computed as BERTScore-F1 with microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fullt

  BraTS-GLI-00000-000
    ROUGE-L          = 0.1550
    METEOR           = 0.3229
    BLEU-4           = 0.0610


config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    RaTEScore(approx)      = 0.7973


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    GREEN(approx)          = 0.7973

Per-patient results saved → evaluation/results.csv

Metric                            Mean       Std
  ROUGE-L                       0.1558    0.0119
  METEOR                        0.2469    0.0534
  BLEU-4                        0.0347    0.0215
  BERTScore-F1                  0.5363    0.0228
  RadGraph-F1                      N/A       N/A
  RaTEScore (approx)            0.7996    0.0066
  GREEN (approx)                0.7996    0.0066

Evaluation complete. Results in evaluation/


## 9. Save all outputs back to Drive

In [11]:
def sync_to_drive(src_name):
    src = Path(WORK_DIR) / src_name
    dst = Path(DRIVE_BRATS_DIR) / src_name
    if not src.exists():
        print(f"  SKIP {src_name} — not found in local storage")
        return
    if dst.exists():
        shutil.rmtree(str(dst)) if dst.is_dir() else dst.unlink()
    if src.is_dir():
        shutil.copytree(str(src), str(dst))
    else:
        shutil.copy2(str(src), str(dst))
    print(f"  Saved: {dst}")

for item in ["reports", "reference_reports", "evaluation"]:
    sync_to_drive(item)

print("\nAll outputs saved to Drive.")

  Saved: /content/drive/MyDrive/Brats/reports
  Saved: /content/drive/MyDrive/Brats/reference_reports
  Saved: /content/drive/MyDrive/Brats/evaluation

All outputs saved to Drive.


## 10. QLoRA Fine-tuning
Fine-tunes only the **language model** layers of Qwen2.5-VL-3B-Instruct using
4-bit quantisation (QLoRA). The vision encoder is frozen — we are adapting the
model's text-generation style to match the reference report vocabulary, not
retraining visual perception.

- 4 patients for training, 1 held out for evaluation
- 2 epochs, LoRA rank 16 on q/k/v/o projections
- Fits on T4 (≈8–10 GB VRAM with gradient checkpointing)

In [12]:
!pip install -q peft bitsandbytes trl
print('peft / bitsandbytes / trl installed.')
### 10a. Load model in 4-bit and apply LoRA
import gc, torch
from pathlib import Path
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Free any GPU memory left from the inference run
gc.collect()
torch.cuda.empty_cache()

MODEL_ID   = 'Qwen/Qwen2.5-VL-3B-Instruct'
WORK_DIR   = '/content/Brats'   # already set from earlier cells

# ── 4-bit quantisation config ─────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,   # saves ~0.4 GB extra
)

print('Loading Qwen2.5-VL-3B-Instruct in 4-bit ...')
model_ft = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',          # single T4 → everything on cuda:0
    trust_remote_code=True,
)
processor_ft = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# prepare_model_for_kbit_training enables gradient checkpointing and
# upcasts LayerNorm weights to float32 for stable training
model_ft = prepare_model_for_kbit_training(
    model_ft, use_gradient_checkpointing=True
)

# ── Freeze the vision encoder ─────────────────────────────────────────────
# Style adaptation is a language problem; the vision encoder already works.
frozen = 0
for name, param in model_ft.named_parameters():
    if 'visual' in name:
        param.requires_grad = False
        frozen += param.numel()
print(f'Vision encoder frozen ({frozen/1e6:.0f}M params).')

# ── LoRA on language-model attention layers only ──────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model_ft = get_peft_model(model_ft, lora_config)
model_ft.print_trainable_parameters()

print(f'GPU memory after model load: '
      f'{torch.cuda.memory_allocated()/1e9:.1f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.1f} GB reserved')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 3.5 MB/s eta 0:00:00
peft / bitsandbytes / trl installed.
Loading Qwen2.5-VL-3B-Instruct in 4-bit ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Vision encoder frozen (335M params).
trainable params: 7,372,800 || all params: 3,761,995,776 || trainable%: 0.1960
GPU memory after model load: 3.1 GB allocated, 7.8 GB reserved


### 10b. Training split and data helpers

In [13]:
import importlib.util, os, warnings
import numpy as np

os.chdir(WORK_DIR)

# ── Import helpers from run_inference.py ─────────────────────────────────
spec = importlib.util.spec_from_file_location('run_inference', f'{WORK_DIR}/run_inference.py')
ri = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ri)   # loads functions without calling main()

# ── Load reference reports (training targets) ─────────────────────────────
def load_ref_reports(ref_dir):
    reports = {}
    for p in sorted(Path(ref_dir).glob('*.txt')):
        raw  = p.read_text(encoding='utf-8')
        text = '\n'.join(l for l in raw.splitlines()
                         if not l.startswith('#')).strip()
        reports[p.stem] = text
    return reports

ref_reports = load_ref_reports(f'{WORK_DIR}/reference_reports')
print(f'Reference reports loaded: {list(ref_reports.keys())}')

# ── Patient split ─────────────────────────────────────────────────────────
all_patients = sorted(
    d for d in Path(f'{WORK_DIR}/brats_data').iterdir()
    if d.is_dir() and d.name.startswith('BraTS-')
)

HELD_OUT     = all_patients[-1]          # last patient held out for eval
train_pats   = [p for p in all_patients if p != HELD_OUT]

print(f'Training  ({len(train_pats)}): {[p.name for p in train_pats]}')
print(f'Held-out  (1):              {HELD_OUT.name}')

# ── Label-masking helper ──────────────────────────────────────────────────
# Mask all tokens before the assistant response with -100 so the loss
# is computed only on the generated text, not on the prompt or video tokens.
def make_labels(input_ids, processor):
    # The response always starts right after '<|im_start|>assistant\n'
    header_ids = processor.tokenizer.encode(
        '<|im_start|>assistant\n', add_special_tokens=False
    )
    ids    = input_ids.tolist()
    labels = [-100] * len(ids)
    n      = len(header_ids)
    # Search from the end — finds the (single) assistant turn
    for i in range(len(ids) - n, -1, -1):
        if ids[i : i + n] == header_ids:
            labels[i + n :] = ids[i + n :]   # unmask response tokens
            break
    return torch.tensor(labels, device=input_ids.device)

Reference reports loaded: ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000', 'BraTS-GLI-00006-000']
Training  (4): ['BraTS-GLI-00000-000', 'BraTS-GLI-00002-000', 'BraTS-GLI-00003-000', 'BraTS-GLI-00005-000']
Held-out  (1):              BraTS-GLI-00006-000


### 10c. Training loop (2 epochs)

In [14]:
from torch.optim import AdamW
from qwen_vl_utils import process_vision_info

# ── Hyper-parameters ─────────────────────────────────────────────────────
NUM_EPOCHS      = 2
LEARNING_RATE   = 2e-4
# With only 4 training patients, accumulate all gradients before stepping
GRAD_ACCUM      = len(train_pats)          # = 4

# Subsample frames for training to reduce VRAM and speed up the loop.
# 12 slices/vol × 4 modalities = 48 frames — enough for style adaptation.
TRAIN_MAX_SLICES = 12

optimizer = AdamW(
    [p for p in model_ft.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=0.01,
)

model_ft.train()
optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    sep = '='*56
    print(f'\n{sep}')
    print(f'  Epoch {epoch+1}/{NUM_EPOCHS}')
    print(sep)
    epoch_loss = 0.0

    for step, patient_dir in enumerate(train_pats):
        pid      = patient_dir.name
        ref_text = ref_reports.get(pid, '')
        if not ref_text:
            print(f'  SKIP {pid} — no reference report')
            continue

        print(f'  [{step+1}/{len(train_pats)}] {pid}')

        # Build frames — uses cached PNGs if already extracted by inference
        # Temporarily lower the slice cap for training to save VRAM
        orig_max = ri.MAX_SLICES_PER_VOL
        ri.MAX_SLICES_PER_VOL = TRAIN_MAX_SLICES
        frame_paths = ri.pack_patient_video(patient_dir)
        ri.MAX_SLICES_PER_VOL = orig_max
        print(f'    {len(frame_paths)} frames')

        messages = [
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'video', 'video': frame_paths, 'fps': 1.0,
                        'min_pixels': ri.FRAME_MIN_PIXELS,
                        'max_pixels': ri.FRAME_MAX_PIXELS,
                    },
                    {'type': 'text', 'text': ri.build_prompt()},
                ],
            },
            {'role': 'assistant', 'content': ref_text},
        ]

        text_in = processor_ft.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        img_in, vid_in = process_vision_info(messages)
        inputs = processor_ft(
            text=[text_in], images=img_in, videos=vid_in,
            padding=True, return_tensors='pt',
        ).to('cuda')

        labels = make_labels(inputs['input_ids'][0], processor_ft).unsqueeze(0)

        outputs = model_ft(**inputs, labels=labels)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM

        print(f'    loss = {loss.item() * GRAD_ACCUM:.4f}')

        # Free activations between patients
        del inputs, outputs, labels
        torch.cuda.empty_cache()

        # Optimizer step after accumulating all patients (or last patient)
        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_pats):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_ft.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            optimizer.zero_grad()
            print(f'    optimizer step after patient {step+1}')

    print(f'  Epoch {epoch+1} mean loss: {epoch_loss / len(train_pats):.4f}')

print('\nFine-tuning complete.')


  Epoch 1/2
  [1/4] BraTS-GLI-00000-000
    [t1c]  BraTS-GLI-00000-000-t1c.nii
           → 12 slices sampled
    [t1n]  BraTS-GLI-00000-000-t1n.nii
           → 12 slices sampled
    [t2f]  BraTS-GLI-00000-000-t2f.nii
           → 12 slices sampled
    [t2w]  BraTS-GLI-00000-000-t2w.nii
           → 12 slices sampled
    48 frames


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


    loss = 2.4592
  [2/4] BraTS-GLI-00002-000
    [t1c]  BraTS-GLI-00002-000-t1c.nii
           → 12 slices sampled
    [t1n]  BraTS-GLI-00002-000-t1n.nii
           → 12 slices sampled
    [t2f]  BraTS-GLI-00002-000-t2f.nii
           → 12 slices sampled
    [t2w]  BraTS-GLI-00002-000-t2w.nii
           → 12 slices sampled
    48 frames
    loss = 2.5666
  [3/4] BraTS-GLI-00003-000
    [t1c]  BraTS-GLI-00003-000-t1c.nii
           → 12 slices sampled
    [t1n]  BraTS-GLI-00003-000-t1n.nii
           → 12 slices sampled
    [t2f]  BraTS-GLI-00003-000-t2f.nii
           → 12 slices sampled
    [t2w]  BraTS-GLI-00003-000-t2w.nii
           → 12 slices sampled
    48 frames
    loss = 2.8810
  [4/4] BraTS-GLI-00005-000
    [t1c]  BraTS-GLI-00005-000-t1c.nii
           → 12 slices sampled
    [t1n]  BraTS-GLI-00005-000-t1n.nii
           → 12 slices sampled
    [t2f]  BraTS-GLI-00005-000-t2f.nii
           → 12 slices sampled
    [t2w]  BraTS-GLI-00005-000-t2w.nii
           → 12 slices sa

### 10d. Save LoRA adapter to Drive

In [15]:
ADAPTER_DIR = f'{WORK_DIR}/qlora_adapter'
Path(ADAPTER_DIR).mkdir(exist_ok=True)

model_ft.save_pretrained(ADAPTER_DIR)
processor_ft.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved locally → {ADAPTER_DIR}')

sync_to_drive('qlora_adapter')
print('Adapter synced to Drive.')

Adapter saved locally → /content/Brats/qlora_adapter
  Saved: /content/drive/MyDrive/Brats/qlora_adapter
Adapter synced to Drive.


### 10e. Generate reports with the fine-tuned model
Loads the base model in bfloat16 (full precision for cleaner output) and
merges the LoRA adapter. Reports go to `reports_finetuned/` so base reports
in `reports/` are preserved for comparison.

In [16]:
import gc
from peft import PeftModel

# Free fine-tuning model before loading inference model
del model_ft
gc.collect()
torch.cuda.empty_cache()
print(f'GPU after clearing ft model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load base model in bfloat16 and attach the adapter
print('Loading base model + LoRA adapter ...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True,
)
model_inf_ft = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_inf_ft.eval()
print('Model ready.')

# ── Inference on all patients (including held-out) ────────────────────────
os.chdir(WORK_DIR)
ft_reports_dir = Path(f'{WORK_DIR}/reports_finetuned')
ft_reports_dir.mkdir(exist_ok=True)

for patient_dir in all_patients:
    pid         = patient_dir.name
    report_path = ft_reports_dir / f'{pid}.txt'

    if report_path.exists():
        print(f'  {pid} — already exists, skipping.')
        continue

    print(f'\n  Generating: {pid}')
    frame_paths = ri.pack_patient_video(patient_dir)  # uses cached PNGs
    report = ri.run_inference(
        model_inf_ft, processor_ft, frame_paths, pid, 'cuda', torch.bfloat16
    )

    with open(report_path, 'w', encoding='utf-8') as fh:
        fh.write(f'# Patient ID : {pid}\n')
        fh.write(f'# Model      : {MODEL_ID} + QLoRA\n')
        fh.write('# ' + '='*62 + '\n\n')
        fh.write(report + '\n')
    print(f'  Saved → {report_path}')

print('\nFine-tuned inference complete.')

GPU after clearing ft model: 2.0 GB
Loading base model + LoRA adapter ...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model ready.

  Generating: BraTS-GLI-00000-000
    [t1c]  BraTS-GLI-00000-000-t1c.nii
           → 48 slices sampled
    [t1n]  BraTS-GLI-00000-000-t1n.nii
           → 48 slices sampled
    [t2f]  BraTS-GLI-00000-000-t2f.nii
           → 48 slices sampled
    [t2w]  BraTS-GLI-00000-000-t2w.nii
           → 48 slices sampled
    Generating report for BraTS-GLI-00000-000 …
  Saved → /content/Brats/reports_finetuned/BraTS-GLI-00000-000.txt

  Generating: BraTS-GLI-00002-000
    [t1c]  BraTS-GLI-00002-000-t1c.nii
           → 48 slices sampled
    [t1n]  BraTS-GLI-00002-000-t1n.nii
           → 48 slices sampled
    [t2f]  BraTS-GLI-00002-000-t2f.nii
           → 48 slices sampled
    [t2w]  BraTS-GLI-00002-000-t2w.nii
           → 48 slices sampled
    Generating report for BraTS-GLI-00002-000 …
  Saved → /content/Brats/reports_finetuned/BraTS-GLI-00002-000.txt

  Generating: BraTS-GLI-00003-000
    [t1c]  BraTS-GLI-00003-000-t1c.nii
           → 48 slices sampled
    [t1n]  BraTS-GLI-0

In [17]:
sync_to_drive('reports_finetuned')
print('Fine-tuned reports saved to Drive.')

  Saved: /content/drive/MyDrive/Brats/reports_finetuned
Fine-tuned reports saved to Drive.


### 10f. Evaluate fine-tuned reports

In [18]:
import importlib.util

os.chdir(WORK_DIR)
spec_ft = importlib.util.spec_from_file_location('evaluate_ft', f'{WORK_DIR}/evaluate.py')
ev_ft   = importlib.util.module_from_spec(spec_ft)
spec_ft.loader.exec_module(ev_ft)

# Point to fine-tuned reports and a separate output directory
ev_ft.REPORTS_DIR = Path(f'{WORK_DIR}/reports_finetuned')
ev_ft.EVAL_DIR    = Path(f'{WORK_DIR}/evaluation_finetuned')
ev_ft.RESULTS_CSV = ev_ft.EVAL_DIR / 'results.csv'

ev_ft.main()


Evaluating 5 patient(s) …



/content/Brats/evaluate.py:190: UserWarning: Could not load YtongXie/RaTEScore (YtongXie/RaTEScore is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`).
  APPROXIMATION: RaTEScore will be computed as BERTScore-F1 with microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.
  warnings.warn(
/content/Brats/evaluate.py:251: UserWarning: Could not load StanfordMIMI/GREEN (StanfordMIMI/GREEN is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`).
  APPROXIMATION: GREEN will be computed as BERTScore-F1 with microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fullt

  BraTS-GLI-00000-000
    ROUGE-L          = 0.1899
    METEOR           = 0.3169
    BLEU-4           = 0.0888


Loading weights:   0%|          | 0/772 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-xlarge-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    RaTEScore(approx)      = 0.7895


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    GREEN(approx)          = 0.7895

Per-patient results saved → /content/Brats/evaluation_finetuned/results.csv

Metric                            Mean       Std
  ROUGE-L                       0.1612    0.0246
  METEOR                        0.2456    0.0444
  BLEU-4                        0.0383    0.0308
  BERTScore-F1                  0.5448    0.0273
  RadGraph-F1                      N/A       N/A
  RaTEScore (approx)            0.7991    0.0111
  GREEN (approx)                0.7991    0.0111

Evaluation complete. Results in /content/Brats/evaluation_finetuned/


### 10g. Base vs fine-tuned — metric comparison

In [19]:
import pandas as pd, numpy as np

base_csv = f'{WORK_DIR}/evaluation/results.csv'
ft_csv   = f'{WORK_DIR}/evaluation_finetuned/results.csv'

base_df = pd.read_csv(base_csv)
ft_df   = pd.read_csv(ft_csv)

metrics = ['rouge_l', 'meteor', 'bleu_4', 'bertscore_f1', 'ratescore', 'green']
labels  = ['ROUGE-L', 'METEOR', 'BLEU-4', 'BERTScore-F1', 'RaTEScore (approx)', 'GREEN (approx)']

print(f'\n{"Metric":<22}  {"Base":>8}  {"Fine-tuned":>10}  {"Δ":>8}')
print('=' * 54)
for col, label in zip(metrics, labels):
    base_val = pd.to_numeric(base_df[col], errors='coerce').mean()
    ft_val   = pd.to_numeric(ft_df[col],   errors='coerce').mean()
    delta    = ft_val - base_val
    arrow    = '↑' if delta > 0.001 else ('↓' if delta < -0.001 else '=')
    print(f'  {label:<20}  {base_val:>8.4f}  {ft_val:>10.4f}  {arrow} {abs(delta):.4f}')
print('=' * 54)

sync_to_drive('evaluation_finetuned')
print('\nFine-tuned evaluation results saved to Drive.')


Metric                      Base  Fine-tuned         Δ
  ROUGE-L                 0.1558      0.1612  ↑ 0.0054
  METEOR                  0.2469      0.2456  ↓ 0.0013
  BLEU-4                  0.0347      0.0383  ↑ 0.0036
  BERTScore-F1            0.5363      0.5448  ↑ 0.0085
  RaTEScore (approx)      0.7996      0.7991  = 0.0005
  GREEN (approx)          0.7996      0.7991  = 0.0005
  Saved: /content/drive/MyDrive/Brats/evaluation_finetuned

Fine-tuned evaluation results saved to Drive.
